# Customer Intent Intelligence
## Notebook 05 — Final Model & Inference

### Objective

Select the final classification model based on the results from previous
notebooks and create a reusable inference pipeline.

The pipeline will:

1. Accept a raw customer query
2. Preprocess the query
3. Generate its representation
4. Predict the banking intent
5. Calculate prediction confidence
6. Flag low-confidence predictions

## 1.Import

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

# Use the same preprocessing the training data was built with.
sys.path.insert(0, os.path.abspath("../src"))
from preprocessing import preprocess_query

## 2. load model results


In [2]:
results = pd.read_csv(
    "../reports/results/all_model_results.csv"
)

results.round(4)

,Model,Validation Macro F1,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,TF-IDF + Logistic Regression,0.8735,0.8912,0.8958,0.8912,0.8915,0.8915
1,TF-IDF + Linear SVM,0.8820,0.8906,0.8944,0.8906,0.8906,0.8906
2,Embeddings + Logistic Regression,0.9028,0.9081,0.9127,0.9081,0.9079,0.9079
3,Embeddings + Linear SVM,0.9261,0.9276,0.9299,0.9276,0.9273,0.9273
4,Embeddings + MLP,0.8930,0.9068,0.9135,0.9068,0.9073,0.9073


## 3.Select final model

In [3]:
best_model_row = results.loc[
    results["Validation Macro F1"].idxmax()
]

best_model_row

Model                  Embeddings + Linear SVM
Validation Macro F1                   0.926058
Accuracy                              0.927597
Macro Precision                       0.929876
Macro Recall                          0.927597
Macro F1                              0.927331
Weighted F1                           0.927331
Name: 3, dtype: object

In [4]:
best_model_name = best_model_row["Model"]

print("Final model:", best_model_name)
print("Selected on Validation Macro F1:",
      round(best_model_row["Validation Macro F1"], 4))
print("--- Held-out test performance ---")
print("Accuracy:", round(best_model_row["Accuracy"], 4))
print("Macro F1:", round(best_model_row["Macro F1"], 4))
print("Weighted F1:", round(best_model_row["Weighted F1"], 4))

Final model: Embeddings + Linear SVM
Selected on Validation Macro F1: 0.9261
--- Held-out test performance ---
Accuracy: 0.9276
Macro F1: 0.9273
Weighted F1: 0.9273


## 4.Load the model

In [5]:
if best_model_name == "Embeddings + MLP":

    classifier = joblib.load(
        "../models/semantic_mlp.pkl"
    )

elif best_model_name == "Embeddings + Linear SVM":

    classifier = joblib.load(
        "../models/semantic_svm.pkl"
    )

elif best_model_name == "Embeddings + Logistic Regression":

    classifier = joblib.load(
        "../models/semantic_logistic_regression.pkl"
    )

elif best_model_name == "TF-IDF + Linear SVM":

    classifier = joblib.load(
        "../models/linear_svm.pkl"
    )

elif best_model_name == "TF-IDF + Logistic Regression":

    classifier = joblib.load(
        "../models/logistic_regression.pkl"
    )

else:

    raise ValueError(
        f"Unknown model: {best_model_name}"
    )

## 6. load the representation model

In [6]:
if "Embeddings" in best_model_name:

    embedding_model = SentenceTransformer(
        "all-MiniLM-L6-v2"
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
if "TF-IDF" in best_model_name:

    vectorizer = joblib.load(
        "../models/tfidf_vectorizer.pkl"
    )

## 7. Preprocessing function

In [8]:
# preprocess_query is imported from src/preprocessing.py (see cell 1).
#
# Defining it again here previously caused train/serve skew: the local copy
# kept digits and skipped tokenization, while the training data had digits
# removed and "cannot" split into "can not".

import inspect

print(inspect.getsource(preprocess_query))

def preprocess_text(text):
    """Normalize a raw customer query into model-ready text."""

    # Convert to lowercase
    text = str(text).lower()

    # Remove non-alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    tokens = word_tokenize(text)

    # Reconstruct text
    return " ".join(tokens)



In [9]:
query = "Why hasn't my card arrived yet?"

preprocess_query(query)

'why hasn t my card arrived yet'

## 8. Create prediction function

In [10]:
def create_features(text):

    processed_text = preprocess_query(text)

    if "Embeddings" in best_model_name:

        features = embedding_model.encode(
            [processed_text]
        )

    else:

        features = vectorizer.transform(
            [processed_text]
        )

    return features

In [11]:
features = create_features(
    "My card hasn't arrived"
)

print(features.shape)

(1, 384)


## 9. Create prediction function

In [12]:
def predict_intent(text):

    features = create_features(text)

    # Models with probability estimates
    if hasattr(classifier, "predict_proba"):

        probabilities = classifier.predict_proba(
            features
        )[0]

        sorted_indices = np.argsort(
            probabilities
        )[::-1]

        top_index = sorted_indices[0]
        second_index = sorted_indices[1]

        top_score = probabilities[top_index]
        second_score = probabilities[second_index]

        top_intent = classifier.classes_[top_index]
        second_intent = classifier.classes_[second_index]

        margin = (
            top_score
            - second_score
        )

        return {
            "intent": top_intent,
            "score": top_score,
            "score_type": "probability",
            "second_intent": second_intent,
            "second_score": second_score,
            "margin": margin
        }

    # SVM
    else:

        scores = classifier.decision_function(
            features
        )

        sorted_indices = np.argsort(
            scores[0]
        )[::-1]

        top_index = sorted_indices[0]
        second_index = sorted_indices[1]

        top_score = scores[0][top_index]
        second_score = scores[0][second_index]

        top_intent = classifier.classes_[top_index]
        second_intent = classifier.classes_[second_index]

        margin = (
            top_score
            - second_score
        )

        return {
            "intent": top_intent,
            "score": top_score,
            "score_type": "decision_score",
            "second_intent": second_intent,
            "second_score": second_score,
            "margin": margin
        }

In [13]:
result = predict_intent(
    "My card hasn't arrived yet"
)

result

{'intent': 'card_arrival',
 'score': np.float64(0.6705081096530265),
 'score_type': 'decision_score',
 'second_intent': 'card_delivery_estimate',
 'second_score': np.float64(-0.3906757217872643),
 'margin': np.float64(1.0611838314402908)}

In [14]:
result = predict_intent(
    "I need help with something"
)

result

{'intent': 'edit_personal_details',
 'score': np.float64(-0.842828719109741),
 'score_type': 'decision_score',
 'second_intent': 'exchange_rate',
 'second_score': np.float64(-0.8702774113462808),
 'margin': np.float64(0.027448692236539785)}

## 10. User-facing classification

In [15]:
# The confidence gate is tuned on the validation split, never on the test set.
#
# A LinearSVC fitted on the sub-training split is scored on validation, so the
# thresholds below are chosen against data the scoring model never saw.

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

train_embeddings = np.load(
    "../data/processed/embeddings/train_embeddings.npy"
)

train_labels = np.asarray(
    pd.read_csv("../data/processed/train.csv")["category"],
    dtype=object
)

X_tr, X_val, y_tr, y_val = train_test_split(
    train_embeddings,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

tuning_model = LinearSVC(
    C=1.0,
    random_state=42
).fit(X_tr, y_tr)

val_scores = tuning_model.decision_function(X_val)

val_order = np.argsort(val_scores, axis=1)[:, ::-1]

val_top = val_scores[
    np.arange(len(val_scores)),
    val_order[:, 0]
]

val_second = val_scores[
    np.arange(len(val_scores)),
    val_order[:, 1]
]

val_margin = val_top - val_second

val_correct = (
    tuning_model.classes_[val_order[:, 0]] == y_val
)

print("Validation accuracy:", round(val_correct.mean(), 4))

Validation accuracy: 0.9245


### Choosing the confidence thresholds\n\nTwo signals indicate an unreliable SVM prediction:\n\n1. A **negative top decision score** — no one-vs-rest classifier claimed the\n   query, so the winner is only the least-rejected intent.\n2. A **small top-2 margin** — the model cannot separate the best two intents.\n\nA prediction is surfaced as confident only when the top score is positive\n**and** the margin clears a threshold. The threshold is the smallest value on\nthe grid for which confident predictions reach the target accuracy below.

In [16]:
TARGET_CLEAR_ACCURACY = 0.975

sweep = []

for candidate in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 1.00]:

    flagged = (
        (val_margin < candidate)
        | (val_top <= 0)
    )

    sweep.append({
        "Margin threshold": candidate,
        "% flagged": 100 * flagged.mean(),
        "Accuracy if clear": val_correct[~flagged].mean(),
        "% errors caught": (
            100
            * (~val_correct[flagged]).sum()
            / (~val_correct).sum()
        )
    })

sweep_df = pd.DataFrame(sweep)

sweep_df.round(4)

,Margin threshold,% flagged,Accuracy if clear,% errors caught
0,0.10,12.6937,0.9697,64.9007
1,0.20,13.2434,0.9718,67.5497
2,0.30,13.9430,0.9756,72.1854
3,0.40,14.8926,0.9789,76.1589
4,0.50,16.5917,0.9802,78.1457
5,0.60,18.2909,0.9835,82.1192
6,0.75,22.2389,0.9884,88.0795
7,1.00,30.3848,0.9943,94.7020


In [17]:
qualifying = sweep_df[
    sweep_df["Accuracy if clear"] >= TARGET_CLEAR_ACCURACY
]

SVM_MARGIN_THRESHOLD = float(
    qualifying["Margin threshold"].min()
)

SVM_REQUIRE_POSITIVE_SCORE = True

# Probability-based models keep a fixed threshold; the current final model is
# margin-based, so this is only used if a probability model is ever selected.
PROBABILITY_THRESHOLD = 0.60

print("Selected margin threshold:", SVM_MARGIN_THRESHOLD)
print("Require positive decision score:", SVM_REQUIRE_POSITIVE_SCORE)

chosen = sweep_df[
    sweep_df["Margin threshold"] == SVM_MARGIN_THRESHOLD
].iloc[0]

print("\nOn validation, with this gate:")
print("  flagged as uncertain:", round(chosen["% flagged"], 1), "%")
print("  accuracy when shown as clear:", round(chosen["Accuracy if clear"], 4))
print("  share of all errors caught:", round(chosen["% errors caught"], 1), "%")

Selected margin threshold: 0.3
Require positive decision score: True

On validation, with this gate:
  flagged as uncertain: 13.9 %
  accuracy when shown as clear: 0.9756
  share of all errors caught: 72.2 %


In [18]:
def classify_query(text):

    result = predict_intent(text)

    if result["score_type"] == "probability":

        if result["score"] < PROBABILITY_THRESHOLD:

            status = "Low confidence"

        else:

            status = "High confidence"

    else:

        # A prediction is only "clear" if some classifier actually claimed the
        # query (positive score) AND the top two intents are well separated.
        weak_score = (
            SVM_REQUIRE_POSITIVE_SCORE
            and result["score"] <= 0
        )

        narrow_margin = (
            result["margin"] < SVM_MARGIN_THRESHOLD
        )

        if weak_score or narrow_margin:

            status = "Ambiguous prediction"

        else:

            status = "Clear prediction"

    return {
        "query": text,
        "intent": result["intent"],
        "score": round(
            result["score"],
            4
        ),
        "score_type": result["score_type"],
        "second_intent": result["second_intent"],
        "margin": round(
            result["margin"],
            4
        ),
        "status": status
    }

In [19]:
classify_query(
    "My card hasn't arrived yet"
)

{'query': "My card hasn't arrived yet",
 'intent': 'card_arrival',
 'score': np.float64(0.6705),
 'score_type': 'decision_score',
 'second_intent': 'card_delivery_estimate',
 'margin': np.float64(1.0612),
 'status': 'Clear prediction'}

In [20]:
classify_query(
    "I need help with something"
)

{'query': 'I need help with something',
 'intent': 'edit_personal_details',
 'score': np.float64(-0.8428),
 'score_type': 'decision_score',
 'second_intent': 'exchange_rate',
 'margin': np.float64(0.0274),
 'status': 'Ambiguous prediction'}

In [21]:
test_queries = [
    "My card hasn't arrived yet",
    "I forgot my PIN",
    "How do I transfer money?",
    "Why was I charged an extra fee?",
    "I want to cancel my transfer",
    "I need help with something"
]

In [22]:
for query in test_queries:

    result = classify_query(query)

    print("-" * 60)
    print("Query:", result["query"])
    print("Intent:", result["intent"])
    print("Score:", result["score"])
    print("Score type:", result["score_type"])
    print("Second intent:", result["second_intent"])
    print("Margin:", result["margin"])
    print("Status:", result["status"])

------------------------------------------------------------
Query: My card hasn't arrived yet
Intent: card_arrival
Score: 0.6705
Score type: decision_score
Second intent: card_delivery_estimate
Margin: 1.0612
Status: Clear prediction
------------------------------------------------------------
Query: I forgot my PIN
Intent: get_physical_card
Score: 0.1589
Score type: decision_score
Second intent: pin_blocked
Margin: 0.2721
Status: Ambiguous prediction
------------------------------------------------------------
Query: How do I transfer money?
Intent: transfer_into_account
Score: 0.2964
Score type: decision_score
Second intent: receiving_money
Margin: 0.746
Status: Clear prediction
------------------------------------------------------------
Query: Why was I charged an extra fee?
Intent: card_payment_fee_charged
Score: 0.0737
Score type: decision_score
Second intent: extra_charge_on_statement
Margin: 0.0448
Status: Ambiguous prediction
--------------------------------------------------

## 11. Batch prediction

In [23]:
def predict_batch(queries):

    results = []

    for query in queries:

        results.append(
            classify_query(query)
        )

    return pd.DataFrame(results)

In [24]:
predict_batch(test_queries)

,query,intent,score,score_type,second_intent,margin,status
0,My card hasn't arrived yet,card_arrival,0.6705,decision_score,card_delivery_estimate,1.0612,Clear prediction
1,I forgot my PIN,get_physical_card,0.1589,decision_score,pin_blocked,0.2721,Ambiguous prediction
2,How do I transfer money?,transfer_into_account,0.2964,decision_score,receiving_money,0.7460,Clear prediction
3,Why was I charged an extra fee?,card_payment_fee_charged,0.0737,decision_score,extra_charge_on_statement,0.0448,Ambiguous prediction
4,I want to cancel my transfer,cancel_transfer,0.7303,decision_score,pending_transfer,1.6510,Clear prediction
5,I need help with something,edit_personal_details,-0.8428,decision_score,exchange_rate,0.0274,Ambiguous prediction


## 12. Check uncertainty distribution on test data

In [25]:
semantic_predictions = pd.read_csv(
    "../reports/results/semantic_predictions.csv"
)

In [26]:
baseline_predictions = pd.read_csv(
    "../reports/results/baseline_predictions.csv"
)

prediction_columns = {
    "Embeddings + MLP": (semantic_predictions, "mlp_prediction"),
    "Embeddings + Linear SVM": (semantic_predictions, "svm_prediction"),
    "Embeddings + Logistic Regression": (semantic_predictions, "lr_prediction"),
    "TF-IDF + Linear SVM": (baseline_predictions, "svm_prediction"),
    "TF-IDF + Logistic Regression": (baseline_predictions, "lr_prediction")
}

source_df, prediction_column = prediction_columns[best_model_name]

final_predictions = source_df[prediction_column]

test_accuracy = (
    final_predictions == source_df["category"]
).mean()

print("Final model:", best_model_name)
print("Test predictions:", len(final_predictions))
print("Test accuracy:", round(test_accuracy, 4))

Final model: Embeddings + Linear SVM
Test predictions: 3080
Test accuracy: 0.9276


## Save final config

In [27]:
final_config = {
    "model_name": best_model_name,
    "embedding_model": (
        "all-MiniLM-L6-v2"
        if "Embeddings" in best_model_name
        else None
    ),
    "probability_threshold": PROBABILITY_THRESHOLD,
    "svm_margin_threshold": SVM_MARGIN_THRESHOLD,
    "svm_require_positive_score": SVM_REQUIRE_POSITIVE_SCORE
}

final_config

{'model_name': 'Embeddings + Linear SVM',
 'embedding_model': 'all-MiniLM-L6-v2',
 'probability_threshold': 0.6,
 'svm_margin_threshold': 0.3,
 'svm_require_positive_score': True}

In [28]:
os.makedirs(
    "../models",
    exist_ok=True
)

joblib.dump(
    final_config,
    "../models/final_config.pkl"
)

['../models/final_config.pkl']

In [29]:
print("=" * 50)
print("FINAL MODEL")
print("=" * 50)

print("Model:", best_model_name)

print(
    "Selected on Validation Macro F1:",
    round(
        best_model_row["Validation Macro F1"],
        4
    )
)

print("-" * 50)
print("Held-out test performance")
print("-" * 50)

print(
    "Accuracy:",
    round(
        best_model_row["Accuracy"],
        4
    )
)

print(
    "Macro F1:",
    round(
        best_model_row["Macro F1"],
        4
    )
)

print(
    "Weighted F1:",
    round(
        best_model_row["Weighted F1"],
        4
    )
)

FINAL MODEL
Model: Embeddings + Linear SVM
Selected on Validation Macro F1: 0.9261
--------------------------------------------------
Held-out test performance
--------------------------------------------------
Accuracy: 0.9276
Macro F1: 0.9273
Weighted F1: 0.9273


# Final Model

The final classifier was selected using Macro F1 as the primary evaluation
metric.

The inference pipeline accepts a raw customer query and:

1. Applies preprocessing.
2. Generates the required feature representation.
3. Predicts one of the BANKING77 intents.
4. Returns the top prediction.
5. Returns a probability for probability-based classifiers.
6. Returns a decision score and top-two margin for SVM.
7. Flags ambiguous predictions when the model does not clearly distinguish
   between competing intents.

The trained model and final configuration are saved for deployment.